# Class probability

In [ ]:
import sys
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

sys.path.insert(0, os.path.abspath(".."))

warnings.filterwarnings("ignore")

SLURRIES = ["G50", "G45", "G40", "G40+IPA"]

In [ ]:
X = pd.read_csv("../../_temp/v1/X.csv", index_col=[0, 1, 2])
Xpred = pd.read_csv("../../_temp/v1/Xpred_2D.csv", index_col=[0, 1, 2])
Delaunay = pd.read_csv("../../_temp/v1/delaunay.Xpred_2D.csv", index_col=[0, 1, 2])

In [ ]:
columns = ["slurry", "cosine_of_contact_angle"]
cos_thetas = X["cosine_of_contact_angle"].reset_index()[columns]
slurry_map = cos_thetas.drop_duplicates().set_index("cosine_of_contact_angle")["slurry"]

slurries = Xpred["cosine_of_contact_angle"].map(slurry_map)
unique_slurries = [s for s in SLURRIES if s in slurries.unique()]

## Plot

In [ ]:
N_COLORS = 8
cmap = mcolors.LinearSegmentedColormap.from_list(
    "gray_blue",
    [mcolors.to_rgba("gray", alpha=0.3), mcolors.to_rgba("tab:blue", alpha=0.8)],
    N=N_COLORS,
)

levels = np.linspace(0, 1, N_COLORS + 1)
norm = mcolors.BoundaryNorm(levels, ncolors=N_COLORS)

### Marginal class probability (phi_1)

In [ ]:
marginal = pd.read_csv(
    "../../benchmarks/v1/gpqr.phi_1.class_marginal.Xpred_2D.csv",
    index_col=["index", "batch", "sample"],
)
marginal = marginal[marginal["target"] == "phi_1"].drop(columns=["target"])
marginal = marginal.groupby(level=["index"]).mean()

In [ ]:
fig, axes = plt.subplots(1, len(slurry_map), sharex=True, sharey=True)

for slurry, ax in zip(unique_slurries, axes):
    ok_pred = slurries == slurry

    this_Xpred = Xpred[ok_pred].to_xarray().to_array().values
    this_prob = marginal[ok_pred.values].values

    delaunay = Delaunay[ok_pred.values].values.reshape(this_Xpred.shape[1:])
    delaunay_masked = np.full_like(delaunay, np.nan, dtype=float)

    ax.contourf(
        this_Xpred[0, ...].squeeze(axis=-1),
        this_Xpred[1, ...].squeeze(axis=-1),
        this_prob.reshape(this_Xpred.shape[1:]).squeeze(axis=-1),
        cmap=cmap,
        norm=norm,
        levels=levels,
    )

    ax.contour(
        this_Xpred[0, ...].squeeze(axis=-1),
        this_Xpred[1, ...].squeeze(axis=-1),
        delaunay.squeeze(axis=-1).astype(float),
        levels=[0.5],
        colors="k",
    )

    ax.set_title(slurry)

fig.tight_layout(rect=[0.02, 0.05, 1.0, 0.8])

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar_ax = fig.add_axes([0.18, 0.82, 0.64, 0.04])
cbar = fig.colorbar(sm, cax=cbar_ax, orientation="horizontal")
cbar.set_label("Marginal probability (phi_1)", labelpad=6)
cbar.ax.xaxis.set_ticks_position("top")
cbar.ax.xaxis.set_label_position("top")
cbar.ax.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False)
cbar.ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"{x:.2f}"))

fig.supxlabel("Rgt")
fig.supylabel("Ca")

fig.show()

### Marginal class probability (phi_3)

In [ ]:
marginal = pd.read_csv(
    "../../benchmarks/v1/gpqr.phi_3.class_marginal.Xpred_2D.csv",
    index_col=["index", "batch", "sample"],
)
marginal = marginal[marginal["target"] == "phi_3"].drop(columns=["target"])
marginal = marginal.groupby(level=["index"]).mean()

In [ ]:
fig, axes = plt.subplots(1, len(slurry_map), sharex=True, sharey=True)

for slurry, ax in zip(unique_slurries, axes):
    ok_pred = slurries == slurry

    this_Xpred = Xpred[ok_pred].to_xarray().to_array().values
    this_prob = marginal[ok_pred.values].values

    delaunay = Delaunay[ok_pred.values].values.reshape(this_Xpred.shape[1:])
    delaunay_masked = np.full_like(delaunay, np.nan, dtype=float)

    ax.contourf(
        this_Xpred[0, ...].squeeze(axis=-1),
        this_Xpred[1, ...].squeeze(axis=-1),
        this_prob.reshape(this_Xpred.shape[1:]).squeeze(axis=-1),
        cmap=cmap,
        norm=norm,
        levels=levels,
    )

    ax.contour(
        this_Xpred[0, ...].squeeze(axis=-1),
        this_Xpred[1, ...].squeeze(axis=-1),
        delaunay.squeeze(axis=-1).astype(float),
        levels=[0.5],
        colors="k",
    )

    ax.set_title(slurry)

fig.tight_layout(rect=[0.02, 0.05, 1.0, 0.8])

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar_ax = fig.add_axes([0.18, 0.82, 0.64, 0.04])
cbar = fig.colorbar(sm, cax=cbar_ax, orientation="horizontal")
cbar.set_label("Marginal probability (phi_3)", labelpad=6)
cbar.ax.xaxis.set_ticks_position("top")
cbar.ax.xaxis.set_label_position("top")
cbar.ax.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False)
cbar.ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"{x:.2f}"))

fig.supxlabel("Rgt")
fig.supylabel("Ca")

fig.show()

## phi_1 vs phi_3

In [ ]:
# phi_1: P(output = type 1), phi_3: P(output = type 2)
def load_marginal_probability(path, target):
    probability = pd.read_csv(path, index_col=["index", "batch", "sample"])
    probability = probability[probability["target"] == target].drop(columns=["target"])
    return probability.groupby(level="index").mean()["marginal_prob"]


prob_type_1 = load_marginal_probability(
    "../../benchmarks/v1/gpqr.phi_1.class_marginal.Xpred_2D.csv", "phi_1"
)
prob_type_2 = load_marginal_probability(
    "../../benchmarks/v1/gpqr.phi_3.class_marginal.Xpred_2D.csv", "phi_3"
)
class_prob = pd.concat(
    [prob_type_1.rename("type_1"), prob_type_2.rename("type_2")], axis=1
).fillna(0.0)

# Hue: relative type-1/type-2 probability.  Saturation: P(type 1 or type 2).
# A point likely to be "other" therefore fades to light grey.
type_1_rgb = np.array(mcolors.to_rgb("tab:red"))
type_2_rgb = np.array(mcolors.to_rgb("tab:blue"))
other_rgb = np.array(mcolors.to_rgb("lightgrey"))
p1 = class_prob["type_1"].to_numpy()
p2 = class_prob["type_2"].to_numpy()
p_type_1_or_2 = np.clip(p1 + p2, 0.0, 1.0)
type_1_share = np.divide(p1, p1 + p2, out=np.full_like(p1, 0.5), where=(p1 + p2) > 0)
type_rgb = type_1_share[:, None] * type_1_rgb + (1 - type_1_share)[:, None] * type_2_rgb
point_rgb = p_type_1_or_2[:, None] * type_rgb + (1 - p_type_1_or_2)[:, None] * other_rgb

fig, axes = plt.subplots(1, len(slurry_map), sharex=True, sharey=True)

for slurry, ax in zip(unique_slurries, axes):
    ok_pred = slurries == slurry
    this_Xpred = Xpred[ok_pred].to_xarray().to_array().values
    shape = this_Xpred.shape[1:-1]
    x = this_Xpred[0, ...].squeeze(axis=-1)
    y = this_Xpred[1, ...].squeeze(axis=-1)

    mesh = ax.pcolormesh(x, y, np.zeros(shape), shading="nearest")
    rgba = np.c_[point_rgb[ok_pred.to_numpy()], np.ones(ok_pred.sum())]
    mesh.set_facecolor(rgba)

    delaunay = Delaunay[ok_pred.to_numpy()].to_numpy().reshape(shape)
    ax.contour(x, y, delaunay.astype(float), levels=[0.5], colors="k")
    ax.set_title(slurry)

fig.tight_layout(rect=[0.02, 0.05, 1.0, 0.8])

# Two-dimensional legend: type 2 -> type 1 horizontally; probability of either
# type vertically.  The bottom row is the "other" colour.
legend_type_1_share = np.linspace(0, 1, 256)[None, :]
legend_type_prob = np.linspace(0, 1, 128)[:, None]
legend_type_rgb = (
    legend_type_1_share[..., None] * type_1_rgb
    + (1 - legend_type_1_share[..., None]) * type_2_rgb
)
legend_rgb = (
    legend_type_prob[..., None] * legend_type_rgb
    + (1 - legend_type_prob[..., None]) * other_rgb
)
legend_ax = fig.add_axes([0.18, 0.85, 0.64, 0.04])
legend_ax.imshow(legend_rgb, origin="lower", extent=[0, 1, 0, 1], aspect="auto")
legend_ax.set_xticks([0, 1], ["Type 2", "Type 1"])
legend_ax.set_yticks([0, 1])

fig.supxlabel("Rgt")
fig.supylabel("Ca")
fig.show()